In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from ultralytics import YOLO
import time
import torch
import torchvision.transforms as transforms

from mmpose.apis import init_model
from models.vitpose.models.backbone.vit import ViT
from models.vitpose.models.detectors.top_down import TopDown
from models.vitpose.models.head.topdown_heatmap_simple_head import TopdownHeatmapSimpleHead
from models.vitpose.models.head import topdown_heatmap_simple_head
from mmpose.registry import MODELS

from vit_pose import direct_inference

/Users/tomaskvapil/miniconda3/envs/Yolo_Tensorflow_VitPose/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/Users/tomaskvapil/miniconda3/envs/Yolo_Tensorflow_VitPose/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/tomaskvapil/miniconda3/envs/Yolo_Tensorflow_VitPose/lib/python3.9/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


# Configuration

In [2]:
FRAMES_DIR = 'frames_0.5'
FRAMES_UPSCALED_DIR = 'frames_0.5_upscaled'
SKELETON_DIR = 'skeletons_yolo_11_new'
SKELETON_UPSCALED_DIR = 'skeletons_yolo_11_upscaled_2'

YOLO_MODEL_DETECTION = 'models/yolo/yolo11x.pt'
SUPER_RESOLUTION_MODEL_PATH = 'models/super_resolution'
VITPOSE_MODEL_PATH = 'models/vitpose'

# Models
_DETECTION_MODEL = None
_VITPOSE_MODEL = None
_DATASET_INFO = None

# --- Super-resolution setup ---
_SR_NET = None
_SR_SCALE = 4
_SR_PB = os.path.join(SUPER_RESOLUTION_MODEL_PATH, "ESPCN_x4.pb")

# Register components

In [3]:
MODELS.register_module(module=ViT, force=True)
MODELS.register_module(module=TopDown, force=True)
MODELS.register_module(module=TopdownHeatmapSimpleHead, force=True)

models.vitpose.models.head.topdown_heatmap_simple_head.TopdownHeatmapSimpleHead

# Upscaling images

In [4]:
def upscale_img_folder(folder_name):

    # Paths
    input_frames = os.path.join(FRAMES_DIR, folder_name)
    output_frames = os.path.join(FRAMES_UPSCALED_DIR, folder_name)

    if not os.path.exists(input_frames):
        print(f"Input folder not found: {input_frames}")
        return

    print("Step 1: Creating upscaled frames...")
    start_time = time.time()

    os.makedirs(output_frames, exist_ok=True)
    files = sorted([f for f in os.listdir(input_frames) if f.endswith('.jpg')])

    init_super_resolution_model()

    for fname in tqdm(files, desc=f"Upscaling {os.path.basename(input_frames)}"):
        input_path = os.path.join(input_frames, fname)
        output_path = os.path.join(output_frames, fname)

        if os.path.exists(output_path):
            continue

        img = cv2.imread(input_path)
        if img is None:
            continue

        # Apply super-resolution or high-quality upscaling
        upscaled_img = apply_super_resolution(img)

        # Save upscaled image
        cv2.imwrite(output_path, upscaled_img)

    upscale_time = time.time() - start_time
    print(f"Upscaling completed in {upscale_time:.2f} seconds")


def init_super_resolution_model():
    """
    Initialize super-resolution model
    """
    global _SR_NET

    if _SR_NET is not None:
        return _SR_NET

    try:
        # Check if dnn_superres is available
        if hasattr(cv2, 'dnn_superres'):
            sr = cv2.dnn_superres.DnnSuperResImpl_create()
        else:
            print("dnn_superres not available in your OpenCV build")
            return None

        # Load model
        sr.readModel(_SR_PB)
        sr.setModel('espcn', _SR_SCALE)

        # Set backend (CPU for compatibility)
        sr.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        sr.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

        print("Super-resolution model initialized successfully")
        _SR_NET = sr
        return sr

    except Exception as e:
        print(f"Failed to initialize super-resolution: {e}")
        return None


def apply_super_resolution(img, target_scale=4):
    """
    Apply super-resolution with guaranteed scale factor
    """
    sr = init_super_resolution_model()
    if sr is None:
        height, width = img.shape[:2]
        return cv2.resize(img, (width * target_scale, height * target_scale), interpolation=cv2.INTER_CUBIC)

    try:
        height, width = img.shape[:2]
        target_height, target_width = height * target_scale, width * target_scale

        # Apply super-resolution
        sr_result = sr.upsample(img)

        # If the SR result isn't exactly the target size, resize to match
        sr_height, sr_width = sr_result.shape[:2]
        if sr_height != target_height or sr_width != target_width:
            sr_result = cv2.resize(sr_result, (target_width, target_height), interpolation=cv2.INTER_LANCZOS4)

        return sr_result

    except Exception as e:
        print(f"Super-resolution failed: {e}")
        height, width = img.shape[:2]
        return cv2.resize(img, (width * target_scale, height * target_scale), interpolation=cv2.INTER_CUBIC)

In [5]:
#upscale_img_folder('002')

# 2 stage detection
yolo model for detecting and vitpose for extracting skeletons

In [16]:
# enhanced_detection_pipeline
def yolo_detection(img, model):
    all_boxes = []
    enhanced_imgs = [img]

    # CLAHE enhancement on upscaled image
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced_img = cv2.merge((l, a, b))
    enhanced_img = cv2.cvtColor(enhanced_img, cv2.COLOR_LAB2BGR)
    enhanced_imgs.append(enhanced_img)

    # Histogram equalization
    yuv = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)
    yuv[:, :, 0] = cv2.equalizeHist(yuv[:, :, 0])
    enhanced2 = cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR)
    enhanced_imgs.append(enhanced2)

    # Gamma correction
    gamma = 0.8
    enhanced3 = np.power(img / 255.0, gamma) * 255.0
    enhanced3 = enhanced3.astype(np.uint8)
    enhanced_imgs.append(enhanced3)

    # Yolo configuration
    configs = [
        {'conf': 0.25, 'imgsz': 1280, 'iou': 0.45},
        {'conf': 0.20, 'imgsz': 1024, 'iou': 0.4},
        {'conf': 0.30, 'imgsz': 1536, 'iou': 0.5},
        {'conf': 0.35, 'imgsz': 1792, 'iou': 0.55},
        {'conf': 0.15, 'imgsz': 2048, 'iou': 0.35},
    ]

    for img_idx, enhanced_img in enumerate(enhanced_imgs):
        if img_idx == 0:
            config_indices = [0, 2]
        elif img_idx == 1:  # CLAHE
            config_indices = [1, 3]
        elif img_idx == 2:  # Histogram equalized
            config_indices = [0, 4]
        else:  # Gamma corrected
            config_indices = [1]

        for config_idx in config_indices:
            try:
                res = model(enhanced_img, verbose=False, **configs[config_idx])
                if res[0].boxes is not None:
                    boxes = res[0].boxes.xyxy.cpu().numpy() 
                    confs = res[0].boxes.conf.cpu().numpy() 
                    all_boxes.append(np.column_stack([boxes, confs]))
            except Exception as e:
                print(f"Detection failed: {e}")
                continue

    if not all_boxes:
        return np.array([])

    combined_boxes = np.vstack(all_boxes)

    return combined_boxes

# process_upscaled_folder
def first_stage_detection(frames_dir, skeleton_dir, model):
    """
    Yolo person detection
    """
    os.makedirs(skeleton_dir, exist_ok=True)
    files = sorted(f for f in os.listdir(frames_dir) if f.endswith('.jpg'))
    all_boxes = []

    total_frames = len(files)
    detected_frames = 0
    total_detections = 0

    for fname in tqdm(files, desc=f"Detecting skeletons in {os.path.basename(frames_dir)}"):
        path = os.path.join(frames_dir, fname)
        img = cv2.imread(path)
        if img is None:
            continue

        boxes = yolo_detection(img, model)
        all_boxes.append(boxes)

## Second stage VitPose

In [19]:
from mmpose.apis import inference_topdown


class VitPoseExtractor:
    def __init__(self, config_file, checkpoint_file, device='cuda'):
        self.model = init_model(config_file, checkpoint_file, device=device)
        
        
    def defextract_keypoints(self, img, boxes):
        
        if len(boxes) == 0:
            return np.array([])
        
        person_results = []
        for box in boxes:
            x1, y1, x2, y2 = box[:4]
            score = box[4] if len(box) > 4 else 1.0
            person_results.append({
                'bbox': np.array([x1, y1, x2, y2, score])
            })
            
        pose_results = inference_topdown(self.model, img, person_results) 
        skeletons = []
        
        for result in pose_results:
            if 'keypoints' in result:
                
                # Get keypoints [17, 3] where 3 = [x, y, confidence]
                keypoints = result['keypoints']
                
                # Take only x, y coordinates [17, 2]
                skeleton = keypoints[:, :2]
                skeletons.append(skeleton)
        
        return np.array(skeletons) if skeletons else np.array([])        

In [17]:
def vitpose_skeleton_extraction(img, boxes, vitpose_model):
    """
    Extract skeletons from detected person bounding boxes using VitPose
    """
    config_file = 'models/vitpose/configs/ViTPose_huge_crowdpose_256x192_without_training.py'
    checkpoint_file = 'models/vitpose/vitpose-h-multi-crowdpose.pth'
    skeletons = []
    
    model = init_model(config_file, checkpoint_file, device='cuda')
    
    for box in boxes:
        x1, y1, x2, y2 = box[:4].astype(int)
        # Crop person from image
        person_crop = img[y1:y2, x1:x2]
        
        keypoints = direct_inference(model, box, img)
        
        
        skeletons.append(keypoints)
    
    return np.array(skeletons) if skeletons else np.array([])

In [18]:
bounding_boxes = first_stage_detection('frames_0.5_upscaled/002', SKELETON_UPSCALED_DIR, YOLO(YOLO_MODEL_DETECTION))

config_file = 'models/vitpose/configs/ViTPose_huge_crowdpose_256x192_without_training.py'
checkpoint_file = 'models/vitpose/vitpose-h-multi-crowdpose.pth'
vit_pose = VitPoseExtractor.__init__(config_file, checkpoint_file, device='cuda')

Detecting skeletons in 002: 100%|██████████| 5/5 [00:53<00:00, 10.67s/it]
